In [21]:
import os
import smtplib
from email.message import EmailMessage

from dotenv import load_dotenv
from openai import AsyncOpenAI
from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax

from agents import Agent, OpenAIChatCompletionsModel, Runner, function_tool, trace

load_dotenv(override=True)
console = Console()
OPENAI_MODEL_NAME = "gpt-5.4-nano"


## Email Setup


In [2]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SERVER = os.getenv("EMAIL_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if not EMAIL_ADDRESS or not EMAIL_SERVER or not EMAIL_APP_PASSWORD:
    print("Email configuration not received")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SERVER and EMAIL_APP_PASSWORD

print("Email setup completed") if USE_EMAIL else print("Email setup failed")


# Test the email
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body)

    with smtplib.SMTP(EMAIL_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)


Email setup completed


#### Setup JSON object for the tool from the function


In [3]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


## Instantiate other LLMs


In [4]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [5]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### OpenAI Compatible Base URLs


In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

#### Create Python Client Library for the LLMs


In [7]:
gemini_client = AsyncOpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

openrouter_client = AsyncOpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)

groq_client = AsyncOpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)


#### Create model object for each LLM (Not required when using OpenAI model - model name can be passed as a string while creating agents)


In [8]:
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite", openai_client=gemini_client
)

openrouter_model = OpenAIChatCompletionsModel("moonshotai/kimi-k2.6", openrouter_client)

groq_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

#### Create agents from the model objects for each LLM


In [9]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(
    name="OpenRouter Sales Agent", instructions=instructions, model=openrouter_model
)
sales_agent3 = Agent(name="Groq Sales Agent", instructions=instructions, model=groq_model)

#### Convert the tools into agents


In [11]:
tool_description = (
    "Use this tool to write a sales email. In the input, just instruct it to write a sales email."
)

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=tool_description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=tool_description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=tool_description)

tools = [tool1, tool2, tool3, send_email_tool]

#### Instantiate the Sales Manager Agent to run the tools from different LLMs


In [12]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(
    name="Sales Manager", instructions=instructions, tools=tools, model=OPENAI_MODEL_NAME
)

#### Run the agent


In [13]:
with trace("Sales Manager Agent with tools created from different models"):
    result = await Runner.run(starting_agent=sales_manager, input=task)


Sent the best-performing draft (from **sales_agent2**) to the user.

In [20]:
console.print(Syntax(code=result.final_output, lexer="markdown", theme="monokai"))

Sent the best-performing draft (from **sales_agent2**) to the user.                                                

In [29]:
console.print(Markdown(markup=result.final_output, style="italic"))

Sent the best-performing draft (from sales_agent2) to the user.